# Example: Processing Lidar Data Directly with Databricks

> Use some Spatial-Utils functions to process points data in a LAZ dataset.

__Notes__

* This example was run using DBR 16.0 on a 3 worker cluster (each with 8 CPUs and 61GB RAM);
* Uses 'spatial-utils-v1' branch with extra packages [route,viz].
* Requires a cluster with product `ST_` spatial sql functions enabled for KeplerGL Viz (may require photon cluster + spatial sql flag enabled).
* For the notebook to render well in github, we add screenshots of the map rendering and charts as well as artificially limit tabular results; when you run the notebook in databricks, you can uncomment and remove limits if desired.
* Requires you to download the [Lidar USA](https://www.lidarusa.com/sample-data.html) and upload this to a Volume. 

---
__Author:__ Mathieu Pelletier <mathieu.pelletier@databricks.com> | _Last Modified:_ 04 APR 2025

### Limitations
- LAS Specification Version 1.4
- Point Data Record Format 3
- Support one file at a time
- Scaled value (x, y, z)
- Header/VLRS in map

## Setup

In [0]:
%pip install -U "databricks-spatial[points,viz] @ git+https://github.com/mathieupelletier-db/mosaic.git@lidar-demo"

In [0]:
# Uninstall previous versions
# %pip uninstall --yes databricks-spatial


In [0]:
dbutils.library.restartPython()

In [0]:
%sql
-- CHANGE THESE VARIABLES AS NEEDED
USE CATALOG mpelletier;
CREATE DATABASE IF NOT EXISTS geospatial;
CREATE VOLUME IF NOT EXISTS geospatial.lidar;

## Download example
Rural transmission lines collected with an HD32 mounted to a DJI M600.

![Transmission Lines](https://www.lidarusa.com/uploads/5/4/1/5/54154851/transmission_1_orig.png)

In [0]:
%sh
file_id = "1Axv8tXKEIVP9zvg91DvaCxNc6SkZukfM"
output = "/Volumes/mpelletier/geospatial/lidar/Velodyne1_001.laz"

curl "https://drive.usercontent.google.com/download?id={$file_id}&confirm=xxx" -o $output


### Custom Spark datasource (format = "las") supports both las and laz file formats.

Options that can be used:
- path: las/laz file path (one file at a time)
- chunkSize: controls the batch size (number of points to read and process in memory)

In [0]:
# Import custom source
import spatial.points

In [0]:
df = spark.read.format("las").option("path", "/Volumes/mpelletier/geospatial/lidar/Velodyne1_001.laz").load()
df.display()

# Explore dataset


### Find boundaries

In [0]:
from pyspark.sql.functions import col, min, max

# Calculate min and max for x, y, z
min_max_values = df.select(
    min(col("x")).alias("x_min"),
    max(col("x")).alias("x_max"),
    min(col("y")).alias("y_min"),
    max(col("y")).alias("y_max"),
    min(col("z")).alias("z_min"),
    max(col("z")).alias("z_max")
)

display(min_max_values)

# Point Cloud Filtering

# Vizualizations

In [0]:
import numpy as np

with laspy.open('/Volumes/mpelletier/geospatial/lidar/Velodyne3_001.laz') as f:
    for points in f.chunk_iterator(10000):
        x_float = np.array(points.x).astype(float)
        print(x_float[0])
        #break


In [0]:
from plotly.offline import init_notebook_mode, plot
import plotly.graph_objs as go

In [0]:
import numpy as np

x_min, x_max = np.percentile(las.x, [0, 100])
y_min, y_max = np.percentile(las.y, [0, 100])
z_min, z_max = np.percentile(las.z, [0, 100])

x_min, x_max, y_min, y_max, z_min, z_max

In [0]:
import numpy as np

# Define the bounding box limits
x_min, x_max = np.percentile(las.x, [0, 100])
y_min, y_max = np.percentile(las.y, [0, 100])
z_min, z_max = np.percentile(las.z, [0, 100])

# Filter points within the bounding box
filtered_indices = np.where(
    (las.x >= x_min) & (las.x <= x_max) &
    (las.y >= y_min) & (las.y <= y_max) &
    (las.z >= z_min) & (las.z <= z_max)
)[0]

# Limit the number of points to display
sample_size = 450000  # Adjust the sample size as needed
indices = np.random.choice(filtered_indices, sample_size, replace=False)

trace1 = go.Scatter3d(
  x=las.x[indices], y=las.y[indices], z=las.z[indices], mode='markers',  
  marker=dict(size=2, color=las.intensity[indices], colorscale='Viridis', opacity=1)
)

data = [trace1]
layout = go.Layout(
  autosize=True, width=1100, height=600,
  margin=dict(l=0, r=0, b=0, t=0),
  scene=dict(xaxis=dict(title="X"), yaxis=dict(title="Y"), zaxis=dict(title="Z"), aspectmode="data")
)

fig = go.Figure(data=data, layout=layout)
displayHTML(plot(fig, filename='3d-scatter-colorscale', output_type='div'))

## Add bounding box to limit x,y,z points

In [0]:
import numpy as np

# Limit the number of points to display
sample_size = 400000  # Adjust the sample size as needed
indices = np.arange(sample_size)

trace1 = go.Scatter3d(
  x=las.x[indices], y=las.y[indices], z=las.z[indices], mode='markers',  
  marker=dict(size=3, color=las.intensity[indices], colorscale='Viridis', opacity=1)
)

data = [trace1]
layout = go.Layout(
  autosize=True, width=1100, height=600,
  margin=dict(l=0, r=0, b=0, t=0),
  scene=dict(xaxis=dict(title="X"), yaxis=dict(title="Y"), zaxis=dict(title="Z"), aspectmode="data")
)

fig = go.Figure(data=data, layout=layout)
displayHTML(plot(fig, filename='3d-scatter-colorscale', output_type='div'))

In [0]:
%pip install nbformat plotly --upgrade

In [0]:
import plotly
print(plotly.__version__)


In [0]:

import pandas as pd
import plotly.express as px
import plotly.io as pio

# Ensure Plotly renders in the notebook
pio.renderers.default = 'notebook'

# Sample 10,000 points from the DataFrame
sample_size = 40000  # Adjust the sample size as needed

# Ensure 'las.x', 'las.y', 'las.z', and 'las.intensity' are numeric lists
las_x = list(map(float, las.x[:sample_size]))
las_y = list(map(float, las.y[:sample_size]))
las_z = list(map(float, las.z[:sample_size]))
las_intensity = list(map(float, las.intensity[:sample_size]))

# Create a DataFrame with the sampled points
df = pd.DataFrame({
    'x': las_x,
    'y': las_y,
    'z': las_z,
    'intensity': las_intensity
})

# Create a 3D scatter plot using Plotly
fig = px.scatter_3d(df, x='x', y='y', z='z', color='intensity')

# Adjust the size of the figure
fig.update_layout(
    width=800,
    height=600
)

# Display the interactive plot
fig.show()

In [0]:
!pip uninstall -y pandas

In [0]:
dbutils.library.restartPython()

In [0]:
# Install Datashader
%pip install datashader pandas numpy --upgrade


In [0]:
pip install bokeh colorcet

In [0]:
%pip install holoviews hvplot bokeh


In [0]:
import datashader as ds
import datashader.transfer_functions as tf
import pandas as pd
import colorcet as cc
import warnings

# Limit the number of points to display
#sample_size = 1000  # Adjust the sample size as needed

# Ensure 'las.x', 'las.y', 'las.z', and 'las.intensity' are numeric lists
las_x = list(map(float, las.x))
las_y = list(map(float, las.y))
las_z = list(map(float, las.z))
las_intensity = list(map(float, las.intensity))

# Create a DataFrame with the limited points
df = pd.DataFrame({
    'x': las_x,
    'y': las_y,
    'z': las_z,
    'intensity': las_intensity
})

# Create a Canvas
canvas = ds.Canvas(plot_width=800, plot_height=600)

# Aggregate the data
warnings.filterwarnings('ignore', r'All-NaN (slice|axis) encountered')
agg = canvas.points(df, 'x', 'y', agg=ds.mean('intensity'))

# Create an image
img = tf.shade(agg)

# Display the image
img

In [0]:
import datashader as ds
import datashader.transfer_functions as tf
import pandas as pd
import colorcet as cc
import warnings
import holoviews as hv
import hvplot.pandas
from bokeh.plotting import output_file, save
from holoviews import opts

# Limit the number of points to display
#sample_size = 1000  # Adjust the sample size as needed

# Ensure 'las.x', 'las.y', 'las.z', and 'las.intensity' are numeric lists
las_x = list(map(float, las.x))
las_y = list(map(float, las.y))
las_z = list(map(float, las.z))
las_intensity = list(map(float, las.intensity))

# Create a DataFrame with the limited points
df = pd.DataFrame({
    'x': las_x,
    'y': las_y,
    'z': las_z,
    'intensity': las_intensity
})

# Create a HoloViews plot with Datashader
points = df.hvplot.scatter(x='x', y='y', z='z', c='intensity', datashade=True, width=800, height=600)

# Rasterize the plot to change the projection
#rasterized_points = hv.operation.datashader.rasterize(points, width=800, height=600)

# Convert the plot to a Bokeh figure
bokeh_figure = hv.render(points.opts(responsive=True).opts(toolbar='above'))

# Save the plot as an HTML file
output_file("plot.html")
save(bokeh_figure)

# Display the HTML file using displayHTML
with open("plot.html", "r") as f:
    html_content = f.read()

displayHTML(html_content)

In [0]:
from bokeh.plotting import figure
from bokeh.embed import components, file_html
from bokeh.resources import CDN

# prepare some data
x = [1, 2, 3, 4, 5]
y = [6, 7, 2, 4, 5]

# create a new plot with a title and axis labels
p = figure(title="simple line example", x_axis_label='x', y_axis_label='y')

# add a line renderer with legend and line thickness
p.line(x, y, legend_label="Temp.", line_width=2)

# create an html document that embeds the Bokeh plot
html = file_html(p, CDN, "my plot1")

# display this html
displayHTML(html)


In [0]:
# import libraries
import numpy as np
import pandas as pd

import holoviews as hv
import hvplot.pandas

# create sample date
df = pd.DataFrame(np.random.rand(50, 2), columns=['col1', 'col2'])
df['col3'] = np.random.randint(0, 2, 50)

# create holoviews scatter plot
hv_scatter = df.hvplot(kind='scatter', x='col1', y='col2', groupby='col3')

# save scatter plot as html
hv.save(hv_scatter, 'hv_scatter.html')

# assign html file to variable
with open('hv_scatter.html', 'r') as html_file:
  html_scatter = html_file.read()

# display scatter plot
displayHTML(html_scatter)


In [0]:
import numpy as np

# Verify the NumPy version
print(np.__version__)

In [0]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

# Create a figure
fig = plt.figure(figsize=(48, 24))

# 3D Scatter Plot
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
np.random.seed(0)
x = list(map(float, las.x))
y = list(map(float, las.y))
z = list(map(float, las.z))
intensity = list(map(float, las.intensity))

# Use intensity for color mapping
scatter = ax1.scatter(x, y, z, c=intensity, cmap='viridis', s=1)
ax1.set_xlabel('X Axis')
ax1.set_ylabel('Y Axis')
ax1.set_zlabel('Z Axis')
ax1.set_title('3D Scatter Plot')

# Add color bar
cbar = fig.colorbar(scatter, ax=ax1, shrink=0.5, aspect=5)
cbar.set_label('Intensity')

# Change the viewing angle
ax1.view_init(elev=30, azim=45)  # Adjust the elevation and azimuth angles as needed

plt.show()

In [0]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

# Create a figure
fig = plt.figure(figsize=(12, 6))

# 3D Scatter Plot
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
np.random.seed(0)
x = np.random.rand(100)
y = np.random.rand(100)
z = np.random.rand(100)
ax1.scatter(x, y, z, c=z, cmap='viridis')
ax1.set_xlabel('X Axis')
ax1.set_ylabel('Y Axis')
ax1.set_zlabel('Z Axis')
ax1.set_title('3D Scatter Plot')

# 3D Surface Plot
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
x = np.linspace(-5, 5, 100)
y = np.linspace(-5, 5, 100)
X, Y = np.meshgrid(x, y)
Z = np.sin(np.sqrt(X**2 + Y**2))
surf = ax2.plot_surface(X, Y, Z, cmap='coolwarm', edgecolor='none')
ax2.set_xlabel('X Axis')
ax2.set_ylabel('Y Axis')
ax2.set_zlabel('Z Axis')
ax2.set_title('3D Surface Plot')

plt.show()



In [0]:
%matplotlib notebook
import matplotlib.pyplot as plt
import numpy as np

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
x, y, z = np.random.rand(3, 100)
ax.scatter(x, y, z)
plt.show()


In [0]:
import plotly.express as px
import pandas as pd

# Limit the number of points to display
sample_size = 1000  # Adjust the sample size as needed
df = pd.DataFrame({
    'x': list(las.x)[:sample_size],
    'y': list(las.y)[:sample_size],
    'z': list(las.z)[:sample_size]
})

fig = px.scatter_3d(df, x='x', y='y', z='z', title='3D Scatter Plot of Lidar Points')
fig.show()

In [0]:
point_format = las.point_format
print(point_format.id)
list(point_format.dimension_names)


In [0]:
# Find out what the point format looks like.
for dimension in las.point_format.dimensions:
    print(dimension.name)

# It looks like we have color data in this file, so we can grab:
blue = las.blue


| Classification Value (bits 0:4) | Meaning                          |
|---------------------------------|----------------------------------|
| 0                               | Created, never classified        |
| 1                               | Unclassified1                    |
| 2                               | Ground                           |
| 3                               | Low Vegetation                   |
| 4                               | Medium Vegetation                |
| 5                               | High Vegetation                  |
| 6                               | Building                         |
| 7                               | Low Point (noise)                |
| 8                               | Model Key-point (mass point)     |
| 9                               | Water                            |
| 10                              | Reserved for ASPRS Definition    |
| 11                              | Reserved for ASPRS Definition    |
| 12                              | Overlap Points2                  |
| 13-31                           | Reserved for ASPRS Definition    |

In [0]:
print(las['X'][0])
print(las['Y'][0])
print(las['Z'][0])
print(las['intensity'][0])
print(las['return_number'][0])
print(las['number_of_returns'][0])
print(las['scan_direction_flag'][0])
print(las['edge_of_flight_line'][0])
print(las['classification'][0])
print(las['synthetic'][0])
print(las['key_point'][0])
print(las['withheld'][0])
print(las['scan_angle_rank'][0])
print(las['user_data'][0])
print(las['point_source_id'][0])
print(las['gps_time'][0])

In [0]:
import numpy as np

z_float = np.array(las.z).astype(float)
z_float

In [0]:
# Check if the 'green' field is present in the 'las' object
if hasattr(las, 'gps_time'):
    z_float = np.array(las.gps_time)
    display(z_float)
else:
    print("The 'green' field is not present in the 'las' object.")

In [0]:
# Print the first record of las
first_record = las[0]
print(first_record)

In [0]:
header_attributes = vars(las.header)
display(header_attributes)

In [0]:
las.vlrs

In [0]:
import matplotlib.pyplot as plt
plt.hist(las.intensity)
plt.title("Histogram of the Intensity Dimension")
plt.show()


In [0]:
from spatial.points import FakeDataSource

spark.read.format("fake").load().show()

# +-----------+----------+-------+-------+
# |       name|      date|zipcode|  state|
# +-----------+----------+-------+-------+
# |Carlos Cobb|2018-07-15|  73003|Indiana|
# | Eric Scott|1991-08-22|  10085|  Idaho|
# | Amy Martin|1988-10-28|  68076| Oregon|
# +-----------+----------+-------+-------+

_Contact us to get spatial sql enabled, if needed; it is in private preview as of this example._ 

In [0]:
%run ../../common/classic_enable_spatial_preview.py

In [0]:
%run ./spatial_sql_flag

In [0]:
from spatial.route import pbf_source
from pyspark.sql import functions as F

from spatial.viz.keplergl import *
from spatial.viz.helpers import *

kviz = KeplerViz(spark)

In [0]:
dbutils.widgets.text("catalog", "default_value", "Catalog")
catalog_value = dbutils.widgets.get("catalog")
display(catalog_value)

In [0]:
# - CATALOG & SCHEMA
catalog = "geospatial_docs"
schema = "pbf_source"

# - TABLES
geometryType = "WKT"
bronzeTable = f"{catalog}.{schema}.andorra_bronze"
silverTable = f"{catalog}.{schema}.andorra_silver"
goldTable = f"{catalog}.{schema}.andorra_gold"

# - PATHS
pbfPath = f"/Volumes/{catalog}/{schema}/osm_files/andorra-latest.osm.pbf"

# - VARIABLES
h3Resolution = 8

## Ingest points cloud data (LAS/LAS or PLY)